In [74]:
import pandas as pd
import re
import os
import numpy as np
import requests

In [75]:
os.getcwd()
#Num format
pd.options.display.float_format = '{:,.2f}'.format

path="scraping_output/final"

In [76]:
def clean_data(df):
    # Lowercase name and size
    df['name'] = df['name'].str.lower()
    df['size'] = df['size'].str.lower()

    # Infer type from name when type is NaN
    apt_kw  = r'departamento|depto|dpto|apto|apartamento|piso|flat|loft|studio|estudio'
    house_kw = r'\bcasa\b|chalet|villa|townhouse|bungalow'
    mask_nan = df['type'].isna()
    df.loc[mask_nan & df['name'].str.contains(apt_kw,   case=False, na=False), 'type'] = 'Apartment'
    df.loc[mask_nan & df['name'].str.contains(house_kw, case=False, na=False), 'type'] = 'House'

    #Keep apartments and houses
    df = df[df['type'].isin(['Apartment', 'House'])]

    def parse_number(val):
        """Parse number string handling dot/comma as thousand or decimal separator."""
        match = re.match(r'^[\d.,]+', str(val).strip())
        if not match:
            return np.nan
        s = match.group()

        has_dot = '.' in s
        has_comma = ',' in s

        if has_dot and has_comma:
            if s.rfind('.') > s.rfind(','):
                # e.g. 1,047.89 → comma=miles, dot=decimal
                s = s.replace(',', '')
            else:
                # e.g. 1.047,89 → dot=miles, comma=decimal
                s = s.replace('.', '').replace(',', '.')
        elif has_dot:
            # dot=thousands if ALL groups after dots have exactly 3 digits
            parts = s.split('.')
            if all(len(p) == 3 for p in parts[1:]):
                s = s.replace('.', '')
        elif has_comma:
            parts = s.split(',')
            if all(len(p) == 3 for p in parts[1:]):
                s = s.replace(',', '')
            else:
                s = s.replace(',', '.')

        return pd.to_numeric(s, errors='coerce')

    # Extract currency prefix (e.g. "USD") from price string and update currency column
    def extract_currency(row):
        val = row['price']
        if pd.isna(val):
            return row['currency'], val
        s = str(val).strip()
        m = re.match(r'^([A-Z]{2,4})\s*([\d.,].*)$', s)
        if m:
            detected = m.group(1)
            price_str = m.group(2)
            # Keep existing currency if already set, otherwise use detected one
            currency = row['currency'] if pd.notna(row['currency']) else detected
            return currency, price_str
        return row['currency'], val

    extracted = df[['price', 'currency']].apply(extract_currency, axis=1, result_type='expand')
    df['currency'] = extracted[0]
    df['price'] = extracted[1]

    # Convert price to numeric: remove "$" then parse
    def parse_price(val):
        if pd.isna(val):
            return np.nan
        return parse_number(str(val).replace('$', ''))

    # Convert size to numeric: remove "m²"/"m2" then parse
    def parse_size(val):
        if pd.isna(val):
            return np.nan
        return parse_number(str(val).replace('m²', '').replace('m2', ''))

    df['price'] = df['price'].apply(parse_price)
    df['size'] = df['size'].apply(parse_size)

    # Drop rows where price or size is NaN
    df = df.dropna(subset=['price', 'size'])

    return df


In [77]:
##Load raw data

raw=pd.read_csv("scraping_output/raw/total_scraping.csv", low_memory=False)
print("Total rows", len(raw))
raw.head()

Total rows 81525


,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation
0,Departamento en Venta en La Carolina,$ 140,NaN,Apartment,70 m²,1.0,2.0,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
1,Departamento en Venta en San Isidro del Inca,$ 69.900,NaN,Apartment,69 m²,3.0,2.0,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
2,Departamento en Venta en Centro De Quito,$ 68.000,NaN,Apartment,106 m²,3.0,2.0,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
3,Oficina en Venta en La Mariscal,$ 140.000,NaN,Accommodation,166 m²,7.0,2.0,-0.20,-78.49,Av. Francisco de Orellana & Avenida 10 de Agos...,Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
4,Departamento en Venta en Nayón,$ 110.000,NaN,Apartment,115 m²,2.0,3.0,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell


In [78]:
raw["currency"].unique()

array([nan, 'BRL'], dtype=object)

In [79]:
print("Data before cleaning")
print(raw.groupby(['country', 'operation']).size().unstack(fill_value=0).to_string())

Data before cleaning
operation    rent  sell
country                
Argentina    3000  3000
Brazil       1200  1200
Chile        3000  3000
Colombia     3000  3000
Costa-rica   3849  4000
Dominicana   1667  3689
Ecuador      3000  3000
El-salvador   666  1321
Guatemala    2764  4000
Honduras     1213  1652
Mexico       5162  7766
Nicaragua    1807  2142
Panama       3427  4000
Peru         3000  3000


In [80]:
## Clean data
clean_scrap=clean_data(raw.copy())
print("Total rows:", len(clean_scrap))
clean_scrap

Total rows: 48511


,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation
0,departamento en venta en la carolina,140.00,NaN,Apartment,70.00,1.0,2.0,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
1,departamento en venta en san isidro del inca,"69,900.00",NaN,Apartment,69.00,3.0,2.0,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
2,departamento en venta en centro de quito,"68,000.00",NaN,Apartment,106.00,3.0,2.0,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
4,departamento en venta en nayón,"110,000.00",NaN,Apartment,115.00,2.0,3.0,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
5,casa en venta en garzota,"350,000.00",NaN,House,622.00,15.0,9.0,-2.15,-79.89,"Ciudadela Ietel, IETEL, Guayaquil, Ecuador",Guayas,Guayaquil,2026-03-19 18:26:04,Properati,Ecuador,sell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81496,"apartamento en alquiler bella vista, 2 hab 2 p...",950.00,NaN,Apartment,129.00,2 Recámaras,3 Baños,NaN,NaN,NaN,NaN,Dominicana,2026-04-23 01:24:14,Encuentra24,Dominicana,rent
81500,alquiler de apartamento en bella vista,"1,150.00",NaN,Apartment,69.00,1 Recámara,1 Baños,NaN,NaN,NaN,NaN,Dominicana,2026-04-23 01:24:14,Encuentra24,Dominicana,rent
81502,apartamento en alquiler - serralles,"1,050.00",NaN,Apartment,53.00,1 Recámara,1 Baños,NaN,NaN,NaN,NaN,Dominicana,2026-04-23 01:24:14,Encuentra24,Dominicana,rent
81513,"alquiler naco torre moderna y bien ubicada, co...","1,450.00",NaN,Apartment,80.00,1 Recámara,1.5 Baños,NaN,NaN,NaN,NaN,Dominicana,2026-04-23 01:24:14,Encuentra24,Dominicana,rent


In [81]:
# currency_map = {
#     'Argentina': 'ARS', 'Brazil': 'BRL', 'Chile': 'CLP',
#     'Colombia': 'COP', 'Costa-rica': 'CRC', 'Ecuador': 'USD',
#     'El-salvador': 'USD', 'Guatemala': 'GTQ', 'Honduras': 'HNL',
#     'Mexico': 'MXN', 'Nicaragua': 'NIO', 'Panama': 'USD', 'Peru': 'PEN'
# }

In [82]:
# # Get exchange rates (exchangerate-api)
# API_KEY="cf53b6293b066860dd66b072"
# url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/latest/USD"
# r = requests.get(url).json()

# if r.get('result') == 'success':
#     rates = r['conversion_rates']
#     last_update = pd.to_datetime(r['time_last_update_utc']).strftime('%Y-%m-%d')

#     exchange_table = pd.DataFrame([
#         {
#             'country': country,
#             'rate': 1.0 if code == 'USD' else (1 / rates[code] if code in rates else None),
#             'last_update': last_update
#         }
#         for country, code in currency_map.items()
#     ])

#     print(exchange_table.to_string(index=False))
# else:
#     print("⚠️ Error al obtener tasas:", r)


# def get_latest_exchange_rate(country_code):
#     """
#     Devuelve el último tipo de cambio disponible (moneda local por 1 USD)
#     usando World Bank API (indicador PA.NUS.FCRF).

#     Parámetros
#     ----------
#     country_code : str  — ISO2 (e.g. 'MX', 'BR', 'CL')

#     Retorna
#     -------
#     dict con keys: country_code, source, date, exchange_rate
#     """
#     country_code = country_code.upper()
#     url = (
#         f"https://api.worldbank.org/v2/country/{country_code}"
#         f"/indicator/PA.NUS.FCRF"
#         f"?format=json&per_page=10&mrv=1"
#     )
#     r = requests.get(url)
#     r.raise_for_status()
#     records = r.json()[1]
#     if not records:
#         raise ValueError(f"Sin datos World Bank para {country_code}")
#     item = records[0]
#     return {
#         "country_code": country_code,
#         "source": "worldbank",
#         "date": item["date"],
#         "exchange_rate": item["value"],
#     }


# # ISO2 codes per country (USD countries get rate=1.0 directly)
# iso2_map = {
#     'Argentina':    'AR',
#     'Brazil':       'BR',
#     'Chile':        'CL',
#     'Colombia':     'CO',
#     'Costa-rica':   'CR',
#     'Ecuador':      'EC',
#     'El-salvador':  'SV',
#     'Guatemala':    'GT',
#     'Honduras':     'HN',
#     'Mexico':       'MX',
#     'Nicaragua':    'NI',
#     'Panama':       'PA',
#     'Peru':         'PE',
# }

# rows = []
# for country, iso2 in iso2_map.items():
#     if iso2 == 'USD':
#         rows.append({
#             'country':              country,
#             'lcu_per_usd':          1.0,    # local currency units per 1 USD (World Bank raw)
#             'rate':                 1.0,    # USD per 1 local unit (used for price conversion)
#             'last_update':          None,
#         })
#     else:
#         wb = get_latest_exchange_rate(iso2)
#         rows.append({
#             'country':              country,
#             'lcu_per_usd':          wb['exchange_rate'],          # World Bank raw value
#             'rate':                 1 / wb['exchange_rate'],      # USD per local unit
#             'last_update':          wb['date'],
#         })

# exchange_table = pd.DataFrame(rows)
# exchange_table

In [83]:
# Get exchange rates from IMF CSV — indicator: US Dollar per domestic currency, Period average
# Fallback per country: Monthly → Quarterly → Annual

COUNTRY_TO_IMF = {
    'Argentina':   'Argentina',
    'Brazil':      'Brazil',
    'Chile':       'Chile',
    'Colombia':    'Colombia',
    'Costa-rica':  'Costa Rica',
    'Dominicana':  'Dominican Republic',
    'Guatemala':   'Guatemala',
    'Honduras':    'Honduras',
    'Mexico':      'Mexico',
    'Nicaragua':   'Nicaragua',
    'Peru':        'Peru',
    'Ecuador':     'Ecuador',
    'El-salvador': 'El Salvador',
    'Panama':      'Panama',
}

imf_csv = pd.read_csv(
    "../exchange_rates_imf.csv",
    on_bad_lines="skip",
    engine="python",
    encoding="utf-8",
    usecols=["COUNTRY", "INDICATOR", "TYPE_OF_TRANSFORMATION", "FREQUENCY", "TIME_PERIOD", "OBS_VALUE"],
)

def parse_imf_period(val):
    if "-M" in str(val):
        return pd.to_datetime(val, format="%Y-M%m")
    if "-Q" in str(val):
        year, q = val.split("-Q")
        return pd.Timestamp(year=int(year), month=(int(q) - 1) * 3 + 1, day=1)
    try:
        return pd.Timestamp(year=int(val), month=1, day=1)
    except Exception:
        return pd.NaT

imf_base = (
    imf_csv
    .loc[
        (imf_csv["INDICATOR"] == "US Dollar per domestic currency") &
        (imf_csv["TYPE_OF_TRANSFORMATION"] == "Period average") &
        (imf_csv["COUNTRY"].isin(COUNTRY_TO_IMF.values()))
    ]
    .assign(
        period=lambda d: d["TIME_PERIOD"].map(parse_imf_period),
        rate=lambda d: pd.to_numeric(d["OBS_VALUE"], errors="coerce"),
    )
    .dropna(subset=["rate", "period"])
)

def last_by_freq(freq):
    sub = imf_base[imf_base["FREQUENCY"] == freq]
    if sub.empty:
        return {}
    return (
        sub.loc[sub.groupby("COUNTRY")["period"].idxmax()]
           .set_index("COUNTRY")[["rate", "period"]]
           .to_dict("index")
    )

freq_data = {f: last_by_freq(f) for f in ["Monthly", "Quarterly", "Annual"]}

rows = []
for country, imf_name in COUNTRY_TO_IMF.items():
    for freq in ["Monthly", "Quarterly", "Annual"]:
        if imf_name in freq_data[freq]:
            rec = freq_data[freq][imf_name]
            rows.append({
                "country":     country,
                "rate":        rec["rate"],
                "last_update": rec["period"].strftime("%Y-%m"),
            })
            break

exchange_table = pd.DataFrame(rows).sort_values("country").reset_index(drop=True)
exchange_table.style.format({"rate": "{:.6f}"})

,country,rate,last_update
0,Argentina,0.000806,2025-01
1,Brazil,0.178978,2025-01
2,Chile,0.001051,2025-01
3,Colombia,0.000264,2025-12
4,Costa-rica,0.002131,2026-03
5,Dominicana,0.015746,2026-01
6,Ecuador,1.000000,2026-03
7,El-salvador,1.000000,2026-03
8,Guatemala,0.130646,2026-03
9,Honduras,0.037709,2026-03


In [84]:
clean_scrap = clean_scrap.merge(
    exchange_table[['country', 'rate', 'last_update']],
    on='country',
    how='left'
)

# Create price_usd:
# Create price_usd:
# - currency == 'USD' → direct price
# - currency == 'UF'  → price * uf_value (CLP) * rate_to_usd del país (CLP→USD)
# - NaN o moneda local → price * rate_to_usd
clean_scrap['price_usd'] = clean_scrap.apply(
    lambda row: row['price'] if row['currency'] == 'USD'
    #UF from Banco Central de Chile
    #https://tinyurl.com/yz5ffy3n
                else row['price'] *39841.72* row['rate'] if row['currency'] == 'UF'
                else row['price'] * row['rate'],
    axis=1
)

#Price per square meter
clean_scrap['price_per_sq_meter'] = clean_scrap['price_usd'] / clean_scrap['size']


clean_scrap

,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation,rate,last_update,price_usd,price_per_sq_meter
0,departamento en venta en la carolina,140.00,NaN,Apartment,70.00,1.0,2.0,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-03,140.00,2.00
1,departamento en venta en san isidro del inca,"69,900.00",NaN,Apartment,69.00,3.0,2.0,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-03,"69,900.00","1,013.04"
2,departamento en venta en centro de quito,"68,000.00",NaN,Apartment,106.00,3.0,2.0,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-03,"68,000.00",641.51
3,departamento en venta en nayón,"110,000.00",NaN,Apartment,115.00,2.0,3.0,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-03,"110,000.00",956.52
4,casa en venta en garzota,"350,000.00",NaN,House,622.00,15.0,9.0,-2.15,-79.89,"Ciudadela Ietel, IETEL, Guayaquil, Ecuador",Guayas,Guayaquil,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-03,"350,000.00",562.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48506,"apartamento en alquiler bella vista, 2 hab 2 p...",950.00,NaN,Apartment,129.00,2 Recámaras,3 Baños,NaN,NaN,NaN,NaN,Dominicana,2026-04-23 01:24:14,Encuentra24,Dominicana,rent,0.02,2026-01,14.96,0.12
48507,alquiler de apartamento en bella vista,"1,150.00",NaN,Apartment,69.00,1 Recámara,1 Baños,NaN,NaN,NaN,NaN,Dominicana,2026-04-23 01:24:14,Encuentra24,Dominicana,rent,0.02,2026-01,18.11,0.26
48508,apartamento en alquiler - serralles,"1,050.00",NaN,Apartment,53.00,1 Recámara,1 Baños,NaN,NaN,NaN,NaN,Dominicana,2026-04-23 01:24:14,Encuentra24,Dominicana,rent,0.02,2026-01,16.53,0.31
48509,"alquiler naco torre moderna y bien ubicada, co...","1,450.00",NaN,Apartment,80.00,1 Recámara,1.5 Baños,NaN,NaN,NaN,NaN,Dominicana,2026-04-23 01:24:14,Encuentra24,Dominicana,rent,0.02,2026-01,22.83,0.29


In [85]:
clean_scrap["currency"].unique()

array([nan, 'USD', 'BRL', 'UF'], dtype=object)

In [95]:
print("\nData after cleaning")
print(clean_scrap.groupby(['country', 'type', 'operation']).size().unstack(fill_value=0).to_string())


Data after cleaning
operation              rent  sell
country     type                 
Argentina   Apartment   475   539
            House       286   504
Brazil      Apartment   907  1000
            House       293   200
Chile       Apartment  1582  2840
Colombia    Apartment   995   519
            House       361   849
Costa-rica  Apartment  1151  1782
            House      1008  1379
Dominicana  Apartment   511  1743
            House        30  1202
Ecuador     Apartment   705   597
            House       678  1280
El-salvador Apartment   110   398
            House        92   904
Guatemala   Apartment   297  1368
            House        89  1176
Honduras    Apartment    45   309
            House         7   661
Mexico      Apartment  1528  1090
            House      1331  4792
Nicaragua   Apartment    22   125
            House       468  1859
Panama      Apartment  1598  2000
            House       891  1973
Peru        Apartment   408   920
            House        95

In [94]:
print("\nsources")
print(clean_scrap.groupby('country')['source'].unique().to_string())


sources
country
Argentina            [Properati]
Brazil             [QuintoAndar]
Chile                     [Yapo]
Colombia             [Properati]
Costa-rica         [Encuentra24]
Dominicana         [Encuentra24]
Ecuador              [Properati]
El-salvador        [Encuentra24]
Guatemala          [Encuentra24]
Honduras           [Encuentra24]
Mexico         [Pincali, iCasas]
Nicaragua          [Encuentra24]
Panama             [Encuentra24]
Peru                 [Properati]


In [87]:
## Save clean_data
clean_scrap.to_csv(path + "/clean_scrap.csv",  index=False, encoding="utf-8-sig")

In [ ]:
print("\nsources")
print(clean_scrap.groupby('country')['source'].unique().to_string())